In [196]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from neuralop.losses.differentiation import FiniteDiff
import os
import scipy.io
from neuralop.models import FNO

device = torch.device("cuda") # Use GPU 

In [197]:
root_dir = '/home/seongwon/AI/neural operator/data/Burgers1dTimeDataset'
os.makedirs(root_dir, exist_ok=True)

# resolution 256

In [198]:
train_data_256 = torch.load(os.path.join(root_dir, 'burgers_train_256.pt'), map_location=device)
test_data_256  = torch.load(os.path.join(root_dir, 'burgers_test_256.pt'),  map_location=device)

x_train_256 = train_data_256['x']  # [1000, 256]
x_train_256 = x_train_256.unsqueeze(1)  # [1000, 1, 256]
y_train_256 = train_data_256['y']  # [1000, 256]
y_train_256 = y_train_256.unsqueeze(1)  # [1000, 1, 256]
x_test_256  = test_data_256['x']   # [200, 256]
x_test_256  = x_test_256.unsqueeze(1)   # [200, 1, 256]
y_test_256  = test_data_256['y']   # [200, 256]
y_test_256  = y_test_256.unsqueeze(1)   # [200, 1, 256]

print(f"train x: {x_train_256.shape}, y: {y_train_256.shape}")
print(f"test  x: {x_test_256.shape},  y: {y_test_256.shape}")

train x: torch.Size([1000, 1, 256]), y: torch.Size([1000, 1, 256])
test  x: torch.Size([200, 1, 256]),  y: torch.Size([200, 1, 256])


In [200]:
model = FNO(n_modes=(16,),
            hidden_channels=64,
            in_channels=1,
            out_channels=1,
            factorization='tucker').to(device)

In [201]:
# 데이터가 무거우니 interative하게 만듦
from torch.utils.data import DataLoader, TensorDataset

train_dataset_256 = TensorDataset(x_train_256, y_train_256)
test_dataset_256 = TensorDataset(x_test_256, y_test_256)

train_loader_256 = DataLoader(train_dataset_256, batch_size=40, shuffle=False)
test_loader_256 = DataLoader(test_dataset_256, batch_size=40, shuffle=False)

In [202]:
def relative_l2_loss(pred, target):
    # 샘플별로 계산 후 평균
    diff = torch.norm(pred - target, dim=-1)
    norm = torch.norm(target, dim=-1)
    return torch.mean(diff / norm)

In [203]:
#학습 시키기 위한 준비

import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5) # learning rate is halved every 100 epochs

In [204]:
device = torch.device('cuda')
model.to(device)

epochs = 500
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x, y in train_loader_256:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        output = model(x)
        loss = relative_l2_loss(output, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    scheduler.step()
    if epoch % 1 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader_256):.6f}")

Epoch [1/500], Loss: 0.755994
Epoch [2/500], Loss: 0.547824
Epoch [3/500], Loss: 0.425971
Epoch [4/500], Loss: 0.378767
Epoch [5/500], Loss: 0.322865
Epoch [6/500], Loss: 0.295408
Epoch [7/500], Loss: 0.295149
Epoch [8/500], Loss: 0.260228
Epoch [9/500], Loss: 0.218891
Epoch [10/500], Loss: 0.212438
Epoch [11/500], Loss: 0.189740
Epoch [12/500], Loss: 0.159969
Epoch [13/500], Loss: 0.159185
Epoch [14/500], Loss: 0.188945
Epoch [15/500], Loss: 0.205012
Epoch [16/500], Loss: 0.174339
Epoch [17/500], Loss: 0.146444
Epoch [18/500], Loss: 0.126531
Epoch [19/500], Loss: 0.152164
Epoch [20/500], Loss: 0.202568
Epoch [21/500], Loss: 0.143611
Epoch [22/500], Loss: 0.124496
Epoch [23/500], Loss: 0.146261
Epoch [24/500], Loss: 0.197355
Epoch [25/500], Loss: 0.150731
Epoch [26/500], Loss: 0.174922
Epoch [27/500], Loss: 0.132446
Epoch [28/500], Loss: 0.196158
Epoch [29/500], Loss: 0.195090
Epoch [30/500], Loss: 0.157292
Epoch [31/500], Loss: 0.131326
Epoch [32/500], Loss: 0.145632
Epoch [33/500], L

In [207]:
# 평가
model.eval()
total_test_loss = 0  # 루프 밖으로

for x, y in test_loader_256:
    x, y = x.to(device), y.to(device)
    with torch.no_grad():
        output = model(x)
        test_loss = relative_l2_loss(output, y)
        total_test_loss += test_loss.item()
    print(f"Batch Test Loss: {test_loss.item():.6f}")

print(f"Average Test Loss: {total_test_loss / len(test_loader_256):.6f}")

Batch Test Loss: 0.093425
Batch Test Loss: 0.088944
Batch Test Loss: 0.095518
Batch Test Loss: 0.088719
Batch Test Loss: 0.111960
Average Test Loss: 0.095713


In [210]:
np.log10(total_test_loss / len(test_loader_256))

np.float64(-1.019027482246623)

# resolution 2048